In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import TruncatedSVD

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("used-car-dealership.csv")

df.head()

,Manufacturer,Model,Year,Cr.,PP,miles
0,Fiat,500 F,1968.0,18500,81.18,12104
1,Volkswagen,Volkswagen 1200,1966.0,35500,177.02,7532
2,Mini,Mini-Cooper 'S',1965.0,44700,251.10,5071
3,Volkswagen,Golf I GTI,1983.0,44500,346.95,33111
4,Mini,Mini-Cooper 'S',1965.0,38000,251.10,49018


In [4]:
print(df.info())
print(df.isnull().sum())

df.columns = df.columns.str.lower()

if 'description' in df.columns:
    df['description'] = df['description'].fillna("No description available")

if 'make' in df.columns:
    df['make'] = df['make'].fillna("Unknown")

if 'model' in df.columns:
    df['model'] = df['model'].fillna("Unknown")

<class 'pandas.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Manufacturer  28 non-null     str    
 1   Model         28 non-null     str    
 2   Year          27 non-null     float64
 3   Cr.           28 non-null     int64  
 4   PP            28 non-null     float64
 5   miles         28 non-null     int64  
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB
None
Manufacturer    0
Model           0
Year            1
Cr.             0
PP              0
miles           0
dtype: int64


In [5]:
numeric_features = ['price', 'mileage', 'year']
categorical_features = ['make', 'model']
text_feature = 'description'

In [6]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

text_transformer = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
        ("text", text_transformer, text_feature)
    ]
)

In [7]:
svd = TruncatedSVD(n_components=100, random_state=42)

In [8]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("svd", svd),
    ("cluster", kmeans)
])

In [10]:
ColumnTransformer([
    ("num", StandardScaler(), ["price"])
])

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_na

In [13]:
df.columns

Index(['manufacturer', 'model', 'year', 'cr.', 'pp', 'miles'], dtype='str')

In [15]:
for col in df.columns:
    print(repr(col))

'manufacturer'
'model'
'year'
'cr.'
'pp'
'miles'


In [18]:
print(model)

KMeans(n_clusters=3, random_state=42)
